# Stage 02 - Vector Embeddings: HuggingFace Embedding Models Integration
### Lecture 21: Local Execution, Normalization, and Batching via LangChain

This notebook demonstrates production HuggingFace embedding integration:
1. Initializing `HuggingFaceEmbeddings` via `langchain-huggingface`.
2. Selecting production models (`sentence-transformers/all-MiniLM-L6-v2`).
3. Normalizing embeddings for efficient dot product similarity search.
4. Batch document embedding (`embed_documents`) and single query embedding (`embed_query`).


In [1]:
# Step 1: Initializing LangChain HuggingFaceEmbeddings
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# Configure model with explicit normalization
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

print("[INFO] Initialized HuggingFaceEmbeddings with normalize_embeddings=True.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[INFO] Initialized HuggingFaceEmbeddings with normalize_embeddings=True.


In [2]:
# Step 2: Query and Document Embeddings Execution
query_text = "What algorithms are used for dense semantic retrieval?"
doc_texts = [
    "Dense passage retrieval uses dual-encoder architectures for vector search.",
    "Inverted indexes utilize term frequency and IDF for exact keyword lookup.",
    "PostgreSQL supports relational queries and ACID transaction management."
]

# Embed single query
query_vector = hf_embeddings.embed_query(query_text)
# Embed document batch
doc_vectors = hf_embeddings.embed_documents(doc_texts)

print(f"[INFO] Query vector dimension: {len(query_vector)}")
print(f"[INFO] Document vectors count: {len(doc_vectors)} (Dimension: {len(doc_vectors[0])})")

# Assertions
assert len(query_vector) == 384, "Query vector dimension must match 384."
assert len(doc_vectors) == 3 and len(doc_vectors[0]) == 384, "Document vectors must match count and dimension."


[INFO] Query vector dimension: 384
[INFO] Document vectors count: 3 (Dimension: 384)


In [3]:
# Step 3: Fast Dot-Product Semantic Ranking (Normalized Vectors)
import numpy as np

# Since vectors are normalized (L2 norm = 1.0), dot product equals cosine similarity
q_vec = np.array(query_vector)
d_mat = np.array(doc_vectors)

# Compute similarity via matrix-vector product
similarity_scores = np.dot(d_mat, q_vec)

print("[INFO] Semantic Retrieval Results:")
for idx, (doc, score) in enumerate(zip(doc_texts, similarity_scores), start=1):
    print(f"  Rank {idx} [Score: {score:.4f}]: {doc}")

best_idx = int(np.argmax(similarity_scores))
print(f"[METRICS] Top Retrieved Document: '{doc_texts[best_idx]}' with score {similarity_scores[best_idx]:.4f}")

# Assertions
assert best_idx == 0, "Dense passage retrieval doc must achieve highest similarity for query."
assert similarity_scores[0] > similarity_scores[2], "Relevant doc must score higher than unrelated PostgreSQL doc."
print("[SUCCESS] HuggingFace embeddings integration and normalized ranking verified.")


[INFO] Semantic Retrieval Results:
  Rank 1 [Score: 0.5795]: Dense passage retrieval uses dual-encoder architectures for vector search.
  Rank 2 [Score: 0.4277]: Inverted indexes utilize term frequency and IDF for exact keyword lookup.
  Rank 3 [Score: 0.2714]: PostgreSQL supports relational queries and ACID transaction management.
[METRICS] Top Retrieved Document: 'Dense passage retrieval uses dual-encoder architectures for vector search.' with score 0.5795
[SUCCESS] HuggingFace embeddings integration and normalized ranking verified.
